# 🤖 Semana 21-22: Machine Learning y SQL Avanzado

**Curso:** Python, SQL, Ciencia de Datos y Análisis de Negocios  
**Duración estimada:** 2 semanas  
**Nivel:** Avanzado

---

## 📋 ¿Qué vas a aprender esta semana?

| Bloque | Temas |
|--------|-------|
| 📉 Regresión lineal | scikit-learn, regularización (Ridge/Lasso), feature selection |
| 🌲 Random Forest | Clasificación, importancia de variables, tuning |
| 🗄️ SQL avanzado | Subconsultas correlacionadas, funciones de ventana, CTEs recursivos |
| 🔗 SQL + Python | Integración análisis SQL → pandas → ML |

---

> 💡 **Prerequisito:** Semanas 1-20 completadas.  
> **Instalaciones necesarias:**
> ```bash
> pip install scikit-learn pandas numpy matplotlib sqlalchemy
> ```

---
## 📉 Bloque 1: Regresión Lineal con scikit-learn

### 📘 Teoría

#### ¿Qué es la regresión lineal?
Modela la relación entre variables de entrada (features) y una salida continua (target):

```
y = β₀ + β₁x₁ + β₂x₂ + ... + βₙxₙ + ε
```

#### El pipeline de scikit-learn
```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge, Lasso

# Preprocesador para columnas mixtas
preprocesador = ColumnTransformer([
    ('num', StandardScaler(),   columnas_numericas),
    ('cat', OneHotEncoder(),    columnas_categoricas),
])

# Pipeline completo
pipeline = Pipeline([
    ('prep',   preprocesador),
    ('modelo', Ridge(alpha=1.0)),
])

pipeline.fit(X_train, y_train)
pred = pipeline.predict(X_test)
```

#### Regularización — evitar sobreajuste

| Método | Penalización | Efecto | Cuándo usar |
|--------|-------------|--------|--------------|
| **Sin regularización** | Ninguna | Puede sobreajustar | Datos simples |
| **Ridge (L2)** | Suma de β² | Reduce todos los coeficientes | Muchas features correladas |
| **Lasso (L1)** | Suma de |β| | Pone coeficientes en 0 | Feature selection automática |
| **ElasticNet** | Mezcla L1+L2 | Combinación | Cuando no sabés cuál elegir |

### 💡 Ejemplo resuelto 1 — Pipeline completo de regresión

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# ✅ Regresión lineal completa con pipeline

np.random.seed(42)
n = 1000
barrios   = ['Palermo','Belgrano','San Telmo','Almagro','Caballito']
df = pd.DataFrame({
    'superficie':  np.random.normal(70, 25, n).clip(20, 200),
    'ambientes':   np.random.choice([1,2,3,4], n, p=[0.1,0.35,0.4,0.15]),
    'antiguedad':  np.random.randint(0, 60, n),
    'piso':        np.random.randint(1, 20, n),
    'cochera':     np.random.choice([0,1], n, p=[0.6,0.4]),
    'barrio':      np.random.choice(barrios, n),
})
mult = {'Palermo':1.3,'Belgrano':1.2,'San Telmo':0.9,'Almagro':1.0,'Caballito':1.05}
df['precio'] = (
    1100*df['superficie'] + 4500*df['ambientes']
    + 12000*df['cochera'] - 180*df['antiguedad']
    + 300*df['piso'] + np.random.normal(0, 8000, n)
) * df['barrio'].map(mult)
df['precio'] = df['precio'].clip(30000, 600000)

# Feature Engineering
df['sup_por_amb'] = df['superficie'] / df['ambientes']
df['es_nuevo']    = (df['antiguedad'] <= 5).astype(int)

# Definir features
num_cols = ['superficie','ambientes','antiguedad','piso','cochera','sup_por_amb','es_nuevo']
cat_cols = ['barrio']
X = df[num_cols + cat_cols]
y = df['precio']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocesador
prep = ColumnTransformer([
    ('num', StandardScaler(),  num_cols),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_cols),
])

# Comparar modelos
modelos = [
    ('Linear Regression', LinearRegression()),
    ('Ridge (α=1)',        Ridge(alpha=1.0)),
    ('Ridge (α=10)',       Ridge(alpha=10.0)),
    ('Lasso (α=100)',      Lasso(alpha=100.0, max_iter=5000)),
    ('ElasticNet',         ElasticNet(alpha=100.0, l1_ratio=0.5, max_iter=5000)),
]

print(f'Dataset: {len(df)} propiedades | Train: {len(X_train)} | Test: {len(X_test)}')
print(f'\n{'Modelo':25} {'MAE':>10} {'R² Test':>10} {'R² CV (5fold)':>15}')
print('-' * 63)

mejor_r2, mejor_nombre, mejor_pipeline = 0, '', None

kf = KFold(n_splits=5, shuffle=True, random_state=42)
for nombre, modelo in modelos:
    pipe = Pipeline([('prep', prep), ('modelo', modelo)])
    pipe.fit(X_train, y_train)
    pred     = pipe.predict(X_test)
    mae      = mean_absolute_error(y_test, pred)
    r2_test  = r2_score(y_test, pred)
    cv_scores= cross_val_score(pipe, X_train, y_train, cv=kf, scoring='r2')
    r2_cv    = cv_scores.mean()
    print(f'  {nombre:23} ${mae:>8,.0f} {r2_test:>10.4f} {r2_cv:>12.4f} ± {cv_scores.std():.3f}')
    if r2_test > mejor_r2:
        mejor_r2, mejor_nombre, mejor_pipeline = r2_test, nombre, pipe

# Análisis del mejor modelo
print(f'\n=== Mejor modelo: {mejor_nombre} (R²={mejor_r2:.4f}) ===')
pred_mejor = mejor_pipeline.predict(X_test)

# Errores por rango de precio
errores = pd.DataFrame({'real': y_test, 'pred': pred_mejor})
errores['error_pct'] = abs(errores['real'] - errores['pred']) / errores['real'] * 100
errores['rango'] = pd.cut(errores['real'],
    bins=[0,100000,200000,300000,600000],
    labels=['<100k','100-200k','200-300k','>300k'])

print('\nError % promedio por rango de precio:')
print(errores.groupby('rango', observed=True)['error_pct']
      .agg(['mean','count']).rename(columns={'mean':'error_%','count':'n'}).round(1).to_string())

# Coeficientes de Lasso (feature selection implícita)
lasso_pipe = Pipeline([('prep', prep), ('modelo', Lasso(alpha=100.0, max_iter=5000))])
lasso_pipe.fit(X_train, y_train)
feature_names = num_cols + list(lasso_pipe.named_steps['prep']
    .named_transformers_['cat'].get_feature_names_out(['barrio']))
coefs = lasso_pipe.named_steps['modelo'].coef_
print('\nCoeficientes Lasso (0 = eliminada):')
for fname, coef in sorted(zip(feature_names, coefs), key=lambda x: abs(x[1]), reverse=True):
    estado = '✅' if coef != 0 else '❌ eliminada'
    print(f'  {fname:30}: {coef:>12.1f}  {estado}')

### ✏️ Ejercicio guiado 1 — Feature Engineering y selección

Mejorá el modelo agregando nuevas variables derivadas y seleccionando las mejores con Lasso.

**Pistas:**
- Variables cuadráticas: `df['sup²'] = df['superficie']**2`
- Interacciones: `df['sup×amb'] = df['superficie'] * df['ambientes']`
- `SelectFromModel(Lasso(...))` selecciona features automáticamente basándose en los coeficientes

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge, Lasso
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import r2_score
import warnings; warnings.filterwarnings('ignore')

# (Usar el df del ejemplo anterior)
np.random.seed(42); n=1000
barrios = ['Palermo','Belgrano','San Telmo','Almagro','Caballito']
df2 = pd.DataFrame({'superficie':np.random.normal(70,25,n).clip(20,200),
    'ambientes':np.random.choice([1,2,3,4],n,p=[0.1,0.35,0.4,0.15]),
    'antiguedad':np.random.randint(0,60,n),'piso':np.random.randint(1,20,n),
    'cochera':np.random.choice([0,1],n,p=[0.6,0.4]),'barrio':np.random.choice(barrios,n)})
mult={'Palermo':1.3,'Belgrano':1.2,'San Telmo':0.9,'Almagro':1.0,'Caballito':1.05}
df2['precio']=(1100*df2['superficie']+4500*df2['ambientes']+12000*df2['cochera']
    -180*df2['antiguedad']+300*df2['piso']+np.random.normal(0,8000,n))*df2['barrio'].map(mult)
df2['precio']=df2['precio'].clip(30000,600000)

# ✏️ Agregá features derivadas:
df2['sup_cuadrada']   = None  # df2['superficie'] ** 2
df2['sup_por_amb']    = None  # df2['superficie'] / df2['ambientes']
df2['interaccion']    = None  # df2['superficie'] * df2['ambientes']
df2['es_nuevo']       = None  # (df2['antiguedad'] <= 5).astype(int)
df2['piso_alto']      = None  # (df2['piso'] >= 10).astype(int)

# ✏️ Definir num_cols_ext con las nuevas features y armar el pipeline:
num_cols_ext = ['superficie','ambientes','antiguedad','piso','cochera']
# Agregá las nuevas columnas a la lista

cat_cols = ['barrio']
X = df2[num_cols_ext + cat_cols].dropna()
y = df2.loc[X.index, 'precio']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# ✏️ Armá el pipeline con ColumnTransformer y Ridge:
prep = ColumnTransformer([('num', StandardScaler(), num_cols_ext),
                           ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_cols)])
# pipe = Pipeline([('prep', prep), ('modelo', Ridge(alpha=1.0))])
# pipe.fit(X_tr, y_tr)
# r2 = r2_score(y_te, pipe.predict(X_te))
# print(f'R² con features extendidas: {r2:.4f}')


<details>
<summary>👀 Ver solución</summary>

```python
df2['sup_cuadrada'] = df2['superficie'] ** 2
df2['sup_por_amb']  = df2['superficie'] / df2['ambientes']
df2['interaccion']  = df2['superficie'] * df2['ambientes']
df2['es_nuevo']     = (df2['antiguedad'] <= 5).astype(int)
df2['piso_alto']    = (df2['piso'] >= 10).astype(int)

num_cols_ext = ['superficie','ambientes','antiguedad','piso','cochera',
                'sup_cuadrada','sup_por_amb','interaccion','es_nuevo','piso_alto']

pipe = Pipeline([('prep', prep), ('modelo', Ridge(alpha=1.0))])
pipe.fit(X_tr, y_tr)
r2 = r2_score(y_te, pipe.predict(X_te))
print(f'R² con features extendidas: {r2:.4f}')  # debería mejorar ~0.005-0.01
```
</details>

### 🚀 Ejercicio libre 1 — Comparativa completa de regresores

Sobre el dataset de viviendas, construí una comparativa exhaustiva:

1. **5 modelos** mínimo: LinearRegression, Ridge, Lasso, ElasticNet, + uno de tu elección
2. **GridSearchCV** para el mejor modelo con al menos 3 hiperparámetros
3. **Curvas de aprendizaje** — ¿el modelo mejora con más datos?
4. **Análisis de residuos** — graficá `(y_real - y_pred)` vs `y_pred` para detectar patrones
5. **Conclusión** — ¿cuál elegirías en producción y por qué?

**Pista para curvas de aprendizaje:**
```python
from sklearn.model_selection import learning_curve
train_sizes, train_scores, val_scores = learning_curve(
    pipeline, X, y, cv=5, train_sizes=np.linspace(0.1, 1.0, 10)
)
```

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, learning_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, r2_score
import warnings; warnings.filterwarnings('ignore')

# 🚀 Tu comparativa completa aquí:


---
## 🌲 Bloque 2: Random Forest — Clasificación

### 📘 Teoría

#### ¿Cómo funciona Random Forest?
```
Dataset original
      │
  ┌───┴──────────────────┐
 Bootstrap           Bootstrap      ← muestras con reemplazo
  sample 1           sample N
      │                   │
  Árbol 1  ...  Árbol N   ← cada árbol ve subset aleatorio de features
      │                   │
  pred_1   ...  pred_N
      │                   │
  └───────────────────────┘
              │
     Votación / promedio   ← predicción final
```

#### Hiperparámetros clave

| Parámetro | Descripción | Valores típicos |
|-----------|-------------|----------------|
| `n_estimators` | Cantidad de árboles | 100-500 |
| `max_depth` | Profundidad máxima del árbol | None, 10, 20 |
| `min_samples_split` | Mínimo de muestras para dividir | 2, 5, 10 |
| `max_features` | Features a considerar en cada split | 'sqrt', 'log2', None |
| `class_weight` | Peso de clases (para desbalance) | 'balanced', None |

#### Métricas de clasificación
```
Precision = TP / (TP + FP)  → de los que predigo positivo, ¿cuántos son realmente positivos?
Recall    = TP / (TP + FN)  → de los positivos reales, ¿cuántos detecto?
F1        = 2 × (P × R) / (P + R)  → balance entre precision y recall
AUC-ROC   → qué tan bien separa las clases (0.5 = azar, 1.0 = perfecto)
```

### 💡 Ejemplo resuelto 2 — Clasificación completa con Random Forest

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, f1_score, accuracy_score)
import warnings; warnings.filterwarnings('ignore')

# ✅ Clasificación: predecir si un cliente va a churnar (abandona el servicio)

np.random.seed(42)
n = 2000

# Dataset de telecomunicaciones sintético
df_tel = pd.DataFrame({
    'antiguedad_meses':  np.random.randint(1, 72, n),
    'llamadas_mes':      np.random.poisson(45, n),
    'datos_gb':          np.random.exponential(5, n).round(1),
    'reclamos_6meses':   np.random.poisson(0.8, n),
    'plan':              np.random.choice(['Básico','Estándar','Premium'], n, p=[0.3,0.5,0.2]),
    'tiene_tv':          np.random.choice([0,1], n, p=[0.6,0.4]),
    'meses_sin_pagar':   np.random.choice([0,1,2,3], n, p=[0.8,0.12,0.05,0.03]),
    'nps_score':         np.random.randint(0, 11, n),
})

# Lógica de churn: influenciada por features reales
prob_churn = (
    0.05
    + 0.003  * df_tel['reclamos_6meses']
    - 0.003  * df_tel['antiguedad_meses']
    + 0.08   * df_tel['meses_sin_pagar']
    - 0.015  * df_tel['nps_score']
    + 0.05   * (df_tel['plan'] == 'Básico').astype(int)
    - 0.03   * df_tel['tiene_tv']
).clip(0.01, 0.95)
df_tel['churn'] = np.random.binomial(1, prob_churn)

print(f'Dataset: {len(df_tel)} clientes')
print(f'Tasa de churn: {df_tel["churn"].mean()*100:.1f}%')
print(f'Desbalance: {df_tel["churn"].value_counts().to_dict()}')

# Preparar features
num_cols = ['antiguedad_meses','llamadas_mes','datos_gb','reclamos_6meses',
            'meses_sin_pagar','nps_score','tiene_tv']
cat_cols = ['plan']
X = df_tel[num_cols + cat_cols]
y = df_tel['churn']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                           random_state=42, stratify=y)

prep = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_cols),
])

modelos_clf = [
    ('Logistic Regression',    LogisticRegression(class_weight='balanced', max_iter=1000)),
    ('Random Forest',          RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)),
    ('Gradient Boosting',      GradientBoostingClassifier(n_estimators=100, random_state=42)),
]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f'\n{'Modelo':25} {'Accuracy':>10} {'F1 macro':>10} {'AUC-ROC':>10} {'CV F1':>10}')
print('-' * 68)

mejor_f1, mejor_pipe_clf = 0, None
for nombre, modelo in modelos_clf:
    pipe = Pipeline([('prep', prep), ('modelo', modelo)])
    pipe.fit(X_tr, y_tr)
    pred     = pipe.predict(X_te)
    pred_proba = pipe.predict_proba(X_te)[:,1]
    acc  = accuracy_score(y_te, pred)
    f1   = f1_score(y_te, pred, average='macro')
    auc  = roc_auc_score(y_te, pred_proba)
    cv   = cross_val_score(pipe, X_tr, y_tr, cv=skf, scoring='f1_macro').mean()
    print(f'  {nombre:23} {acc:>10.3f} {f1:>10.3f} {auc:>10.3f} {cv:>10.3f}')
    if f1 > mejor_f1:
        mejor_f1, mejor_pipe_clf = f1, pipe

# Reporte detallado del RF
rf_pipe = Pipeline([('prep', prep),
    ('modelo', RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42))])
rf_pipe.fit(X_tr, y_tr)
pred_rf = rf_pipe.predict(X_te)

print('\n=== Reporte Random Forest ===')
print(classification_report(y_te, pred_rf, target_names=['No churn','Churn']))

print('=== Matriz de confusión ===')
cm = confusion_matrix(y_te, pred_rf)
print(f'  TN={cm[0,0]:4}  FP={cm[0,1]:4}')
print(f'  FN={cm[1,0]:4}  TP={cm[1,1]:4}')
tn,fp,fn,tp = cm.ravel()
print(f'  Costo negocio: {fn} clientes que se van sin que los retengamos')

# Feature importance
rf_model     = rf_pipe.named_steps['modelo']
feature_names= num_cols + list(rf_pipe.named_steps['prep']
    .named_transformers_['cat'].get_feature_names_out(['plan']))
importancias  = pd.Series(rf_model.feature_importances_, index=feature_names)
print('\n=== Feature Importance ===')
for feat, imp in importancias.sort_values(ascending=False).items():
    barra = '█' * int(imp * 100)
    print(f'  {feat:25}: {imp:.4f}  {barra}')

### ✏️ Ejercicio guiado 2 — Umbral de clasificación y curva ROC

El umbral por defecto de clasificación es 0.5, pero puede no ser óptimo para todos los casos de negocio.

**Contexto:** en el problema de churn, un falso negativo (no detectar que un cliente se va) puede costar más que un falso positivo (llamar a alguien que no se iba a ir).

**Pistas:**
- `model.predict_proba(X)[:,1]` retorna la probabilidad de la clase positiva
- Probá umbrales de 0.3 a 0.7 en pasos de 0.05
- `precision_recall_curve` calcula la curva completa de una vez

In [ ]:
import numpy as np
from sklearn.metrics import precision_recall_curve, f1_score, confusion_matrix

# (Requiere rf_pipe y X_te, y_te del ejemplo anterior)

# ✏️ Obtener probabilidades
proba = rf_pipe.predict_proba(X_te)[:, 1]

# ✏️ Probar distintos umbrales
print('Umbral | Precision | Recall | F1    | FN (clientes perdidos)')
print('-' * 65)
for umbral in np.arange(0.2, 0.71, 0.05):
    pred_umbral = (proba >= umbral).astype(int)
    # Tu código aquí: calcular precision, recall, f1, FN
    pass

# ✏️ Encontrar el umbral que maximiza F1
# precisions, recalls, thresholds = precision_recall_curve(y_te, proba)
# f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-8)
# umbral_optimo = thresholds[np.argmax(f1_scores[:-1])]
# print(f'\nUmbral óptimo para F1: {umbral_optimo:.3f}')


<details>
<summary>👀 Ver solución</summary>

```python
from sklearn.metrics import precision_score, recall_score

for umbral in np.arange(0.2, 0.71, 0.05):
    pred_umbral = (proba >= umbral).astype(int)
    prec = precision_score(y_te, pred_umbral, zero_division=0)
    rec  = recall_score(y_te, pred_umbral)
    f1   = f1_score(y_te, pred_umbral)
    cm   = confusion_matrix(y_te, pred_umbral)
    fn   = cm[1,0]
    print(f'{umbral:.2f}   | {prec:.3f}     | {rec:.3f}  | {f1:.3f} | {fn}')

# Umbral óptimo
precisions, recalls, thresholds = precision_recall_curve(y_te, proba)
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-8)
umbral_optimo = thresholds[np.argmax(f1_scores[:-1])]
print(f'Umbral óptimo para F1: {umbral_optimo:.3f}')
```
</details>

### 🚀 Ejercicio libre 2 — Pipeline de churn completo

Construí el pipeline de predicción de churn más robusto que puedas:

1. **Feature Engineering** — al menos 3 variables nuevas derivadas (ratio, interacciones, flags)
2. **Manejo del desbalance** — probá `class_weight='balanced'` y `SMOTE` (si instalás `imbalanced-learn`)
3. **GridSearchCV** sobre Random Forest con al menos 4 hiperparámetros
4. **Umbral óptimo** — usá la curva precision-recall para encontrar el mejor
5. **Interpretación de negocio** — si el costo de retener un cliente vale $200 y el costo de perderlo es $1000, ¿qué umbral maximiza el beneficio económico?
6. **Reporte ejecutivo** en Markdown con recomendaciones concretas

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
import warnings; warnings.filterwarnings('ignore')

# 🚀 Tu pipeline de churn aquí:


---
## 🗄️ Bloque 3: SQL Avanzado — Funciones de Ventana y CTEs

### 📘 Teoría

#### Funciones de Ventana (Window Functions)
Permiten calcular valores sobre un conjunto de filas relacionadas **sin colapsar el resultado** como hace GROUP BY.

```sql
función() OVER (
    PARTITION BY columna   -- divide en grupos
    ORDER BY columna       -- ordena dentro del grupo
    ROWS BETWEEN ...       -- define el rango de filas
)
```

#### Funciones disponibles

| Función | Descripción | Ejemplo de uso |
|---------|-------------|----------------|
| `ROW_NUMBER()` | Número de fila dentro del grupo | Ranking sin empates |
| `RANK()` | Igual que ROW_NUMBER pero con empates | Ranking con empates |
| `DENSE_RANK()` | Como RANK sin saltar números | |
| `LAG(col, n)` | Valor de n filas anteriores | Variación vs mes anterior |
| `LEAD(col, n)` | Valor de n filas siguientes | Predicción simple |
| `SUM() OVER` | Suma acumulada o por grupo | Running total |
| `AVG() OVER` | Promedio móvil | Moving average |
| `NTILE(n)` | Divide en n cuantiles iguales | Cuartiles, deciles |

#### CTEs Recursivos
```sql
WITH RECURSIVE jerarquia AS (
    -- Caso base
    SELECT id, nombre, jefe_id, 0 AS nivel
    FROM empleados WHERE jefe_id IS NULL
    UNION ALL
    -- Paso recursivo
    SELECT e.id, e.nombre, e.jefe_id, j.nivel + 1
    FROM empleados e
    JOIN jerarquia j ON e.jefe_id = j.id
)
SELECT * FROM jerarquia ORDER BY nivel, nombre;
```

### 💡 Ejemplo resuelto 3 — Análisis completo con window functions

In [ ]:
import sqlite3
import pandas as pd
import numpy as np

# ✅ Window functions y CTEs avanzados

conn = sqlite3.connect(':memory:')
conn.executescript("""
    CREATE TABLE empleados (
        id INTEGER PRIMARY KEY, nombre TEXT, depto TEXT,
        salario REAL, jefe_id INTEGER, fecha_ingreso TEXT
    );
    CREATE TABLE ventas (
        id INTEGER PRIMARY KEY, vendedor_id INTEGER,
        fecha TEXT, monto REAL, region TEXT
    );
""")

# Empleados con jerarquía
empleados = [
    (1,'CEO',         'Dirección', 200000, None,  '2010-01-01'),
    (2,'Dir. Ventas', 'Ventas',    120000, 1,     '2012-03-15'),
    (3,'Dir. IT',     'IT',        130000, 1,     '2011-06-01'),
    (4,'Ger. Ventas', 'Ventas',     80000, 2,     '2015-01-20'),
    (5,'Vendedor 1',  'Ventas',     55000, 4,     '2018-05-10'),
    (6,'Vendedor 2',  'Ventas',     58000, 4,     '2019-02-14'),
    (7,'Dev Senior',  'IT',         90000, 3,     '2014-09-01'),
    (8,'Dev Junior',  'IT',         65000, 7,     '2021-07-15'),
    (9,'Analista',    'IT',         70000, 3,     '2020-03-01'),
    (10,'RRHH',       'Admin',      60000, 1,     '2013-11-01'),
]
conn.executemany('INSERT INTO empleados VALUES (?,?,?,?,?,?)', empleados)

# Ventas mensuales por vendedor
np.random.seed(42)
vendedores = [5, 6]
fechas = pd.date_range('2023-01-01', '2024-12-31', freq='MS')
ventas_data = []
vid = 1
for vid_emp in vendedores:
    base = 50000 if vid_emp == 5 else 42000
    for fecha in fechas:
        monto = base + np.random.normal(0, 8000)
        region = np.random.choice(['Norte','Sur','Centro'])
        ventas_data.append((vid, vid_emp, fecha.strftime('%Y-%m-%d'), max(monto,5000), region))
        vid += 1
conn.executemany('INSERT INTO ventas VALUES (?,?,?,?,?)', ventas_data)
conn.commit()

# ── 1. Ranking de salarios por departamento ──
print('=== Ranking salarial por departamento ===')
q1 = """
    SELECT
        nombre, depto, salario,
        RANK()       OVER (PARTITION BY depto ORDER BY salario DESC) AS rank_depto,
        DENSE_RANK() OVER (ORDER BY salario DESC)                    AS rank_global,
        ROUND(salario * 100.0 / SUM(salario) OVER (PARTITION BY depto), 1) AS pct_depto
    FROM empleados
    ORDER BY depto, rank_depto
"""
df_rank = pd.read_sql(q1, conn)
print(df_rank.to_string(index=False))

# ── 2. Variación mensual de ventas con LAG ──
print('\n=== Variación mensual de ventas (LAG) ===')
q2 = """
    WITH ventas_mensuales AS (
        SELECT
            vendedor_id,
            strftime('%Y-%m', fecha) AS mes,
            SUM(monto)               AS total
        FROM ventas
        GROUP BY vendedor_id, mes
    )
    SELECT
        v.mes, e.nombre,
        ROUND(v.total, 0)                                          AS total,
        ROUND(LAG(v.total) OVER (PARTITION BY v.vendedor_id ORDER BY v.mes), 0) AS mes_anterior,
        ROUND((v.total - LAG(v.total) OVER (PARTITION BY v.vendedor_id ORDER BY v.mes))
              * 100.0 / LAG(v.total) OVER (PARTITION BY v.vendedor_id ORDER BY v.mes), 1) AS var_pct
    FROM ventas_mensuales v
    JOIN empleados e ON v.vendedor_id = e.id
    ORDER BY e.nombre, v.mes
    LIMIT 12
"""
df_lag = pd.read_sql(q2, conn)
print(df_lag.to_string(index=False))

# ── 3. Media móvil 3 meses ──
print('\n=== Media móvil 3 meses (AVG OVER ROWS) ===')
q3 = """
    WITH mensuales AS (
        SELECT vendedor_id, strftime('%Y-%m', fecha) AS mes, SUM(monto) AS total
        FROM ventas GROUP BY vendedor_id, mes
    )
    SELECT
        e.nombre, m.mes,
        ROUND(m.total, 0)          AS total,
        ROUND(AVG(m.total) OVER (
            PARTITION BY m.vendedor_id
            ORDER BY m.mes
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 0)                      AS ma3,
        ROUND(SUM(m.total) OVER (
            PARTITION BY m.vendedor_id
            ORDER BY m.mes
        ), 0)                      AS acumulado
    FROM mensuales m JOIN empleados e ON m.vendedor_id = e.id
    ORDER BY e.nombre, m.mes LIMIT 16
"""
df_ma = pd.read_sql(q3, conn)
print(df_ma.to_string(index=False))

# ── 4. CTE Recursivo: jerarquía organizacional ──
print('\n=== Jerarquía organizacional (CTE recursivo) ===')
q4 = """
    WITH RECURSIVE org AS (
        SELECT id, nombre, depto, salario, jefe_id, 0 AS nivel,
               nombre AS camino
        FROM empleados WHERE jefe_id IS NULL
        UNION ALL
        SELECT e.id, e.nombre, e.depto, e.salario, e.jefe_id,
               org.nivel + 1,
               org.camino || ' → ' || e.nombre
        FROM empleados e JOIN org ON e.jefe_id = org.id
    )
    SELECT
        SUBSTR('          ', 1, nivel*2) || nombre AS jerarquia,
        depto, salario, nivel
    FROM org ORDER BY camino
"""
df_org = pd.read_sql(q4, conn)
for _, row in df_org.iterrows():
    print(f"  {'  '*row['nivel']}{row['jerarquia']:30} {row['depto']:10} ${row['salario']:>9,.0f}")

conn.close()

### ✏️ Ejercicio guiado 3 — NTILE y subconsultas correlacionadas

Usá `NTILE` para segmentar clientes por deciles de gasto y subconsultas correlacionadas para análisis avanzados.

**Pistas:**
- `NTILE(10) OVER (ORDER BY gasto_total DESC)` → decil 1 = top 10%, decil 10 = bottom 10%
- Una subconsulta correlacionada referencia columnas de la query exterior
- `EXISTS (SELECT 1 FROM ...)` es más eficiente que `IN (SELECT ...)` para tablas grandes

In [ ]:
import sqlite3, pandas as pd, numpy as np

conn2 = sqlite3.connect(':memory:')
conn2.executescript("""
    CREATE TABLE clientes (id INTEGER PRIMARY KEY, nombre TEXT, ciudad TEXT, segmento TEXT);
    CREATE TABLE pedidos  (id INTEGER PRIMARY KEY, cliente_id INTEGER, fecha TEXT, total REAL, estado TEXT);
""")
np.random.seed(1)
n_cli=200; n_ped=800
ciudades=['AMBA','Córdoba','Rosario']
segmentos=['Premium','Standard','Básico']
conn2.executemany('INSERT INTO clientes VALUES (?,?,?,?)',
    [(i,f'Cli_{i}',np.random.choice(ciudades),np.random.choice(segmentos,p=[0.2,0.5,0.3]))
     for i in range(1,n_cli+1)])
fechas=pd.date_range('2023-01-01','2024-12-31',periods=n_ped)
conn2.executemany('INSERT INTO pedidos VALUES (?,?,?,?,?)',
    [(i,np.random.randint(1,n_cli+1),fechas[i-1].strftime('%Y-%m-%d'),
      round(np.random.exponential(500),2),
      np.random.choice(['entregado','pendiente','cancelado'],p=[0.75,0.15,0.10]))
     for i in range(1,n_ped+1)])
conn2.commit()

# ✏️ Query 1: Segmentar clientes en deciles por gasto total
print('=== Deciles de gasto ===')
q_deciles = """
    -- Tu query aquí usando NTILE(10) OVER (ORDER BY gasto DESC)
    SELECT 'completar' AS resultado
"""
# print(pd.read_sql(q_deciles, conn2).head(10).to_string())

# ✏️ Query 2: Clientes cuyo último pedido fue hace más de 90 días (subconsulta correlacionada)
print('\n=== Clientes sin compras recientes ===')
q_inactivos = """
    -- Pista: WHERE NOT EXISTS (SELECT 1 FROM pedidos p
    --         WHERE p.cliente_id = c.id AND p.fecha > date('now', '-90 days'))
    SELECT 'completar' AS resultado
"""
# print(pd.read_sql(q_inactivos, conn2).to_string())

# ✏️ Query 3: Para cada cliente, traer el monto de su pedido más grande
print('\n=== Pedido máximo por cliente ===')
q_max = """
    -- Pista: subconsulta correlacionada en SELECT
    -- SELECT c.nombre, (SELECT MAX(p.total) FROM pedidos p WHERE p.cliente_id = c.id) AS max_pedido
    SELECT 'completar' AS resultado
"""
# print(pd.read_sql(q_max, conn2).head(5).to_string())

conn2.close()
print('Descomentá las queries para ver los resultados')


<details>
<summary>👀 Ver solución</summary>

```python
# Query 1 — Deciles
q_deciles = """
    WITH gasto AS (
        SELECT cliente_id, SUM(total) as gasto_total
        FROM pedidos WHERE estado != 'cancelado'
        GROUP BY cliente_id
    )
    SELECT c.nombre, c.segmento, ROUND(g.gasto_total,0) AS gasto,
           NTILE(10) OVER (ORDER BY g.gasto_total DESC) AS decil
    FROM clientes c JOIN gasto g ON c.id = g.cliente_id
    ORDER BY decil, gasto DESC
"""

# Query 2 — Inactivos
q_inactivos = """
    SELECT c.nombre, c.ciudad,
           (SELECT MAX(fecha) FROM pedidos p WHERE p.cliente_id = c.id) AS ultima_compra
    FROM clientes c
    WHERE NOT EXISTS (
        SELECT 1 FROM pedidos p
        WHERE p.cliente_id = c.id AND p.fecha > '2024-09-01'
    )
    ORDER BY ultima_compra DESC
"""

# Query 3 — Máximo pedido
q_max = """
    SELECT c.nombre,
           (SELECT MAX(p.total) FROM pedidos p WHERE p.cliente_id = c.id) AS max_pedido,
           (SELECT COUNT(*) FROM pedidos p WHERE p.cliente_id = c.id AND estado='entregado') AS pedidos_ok
    FROM clientes c
    ORDER BY max_pedido DESC LIMIT 10
"""
```
</details>

### 🚀 Ejercicio libre 3 — Análisis de ventas con window functions

Construí una BD de e-commerce con ventas de 2 años y respondé estas preguntas **solo con SQL**:

1. **Crecimiento YoY** — variación porcentual de ventas mes a mes comparado con el mismo mes del año anterior (`LAG` con offset 12)
2. **Top 3 vendedores por trimestre** — `ROW_NUMBER()` + filtro en CTE
3. **Clientes en riesgo de churn** — compraron en Q1 2023 pero no en Q2 2023 (subconsulta correlacionada)
4. **Percentil 90 de ticket por categoría** — `NTILE(10)` o percentil con ventana
5. **Suma acumulada de ventas por vendedor** — `SUM() OVER (PARTITION BY ... ORDER BY ...)` reseteándose cada año

In [ ]:
import sqlite3, pandas as pd, numpy as np

# 🚀 Tu BD y queries aquí:

# Crear BD:


# Query 1 - YoY:


# Query 2 - Top 3 por trimestre:


# Query 3 - Churn:


# Query 4 - Percentil 90:


# Query 5 - Acumulado:


---
## 🔗 Bloque 4: Integración SQL → Pandas → ML

### 📘 Descripción

En la vida real, los datos viven en bases de datos SQL.  
El pipeline completo es:

```
Base de datos SQL
       │
  pd.read_sql()  ← query con window functions y CTEs
       │
  DataFrame pandas
       │
  Limpieza + Feature Engineering
       │
  scikit-learn Pipeline
       │
  Modelo entrenado
       │
  Predicciones → guardar en BD con to_sql()
```

### 💡 Ejemplo resuelto 4 — Pipeline end-to-end: SQL → ML → Resultados en BD

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import warnings; warnings.filterwarnings('ignore')

# ✅ Pipeline completo: SQL → Feature Engineering → ML → Guardar predicciones

# ── 1. Crear BD con datos de clientes ──
engine = create_engine('sqlite:///:memory:', echo=False)

np.random.seed(42)
n_cli = 500
clientes_df = pd.DataFrame({
    'cliente_id':      range(1, n_cli+1),
    'nombre':          [f'Cliente_{i}' for i in range(1,n_cli+1)],
    'segmento':        np.random.choice(['Premium','Standard','Básico'],n_cli,p=[0.2,0.5,0.3]),
    'ciudad':          np.random.choice(['AMBA','Córdoba','Rosario','Mendoza'],n_cli),
    'fecha_registro':  pd.date_range('2020-01-01',periods=n_cli,freq='D').strftime('%Y-%m-%d'),
})
pedidos_df = pd.DataFrame({
    'pedido_id':   range(1, 2001),
    'cliente_id':  np.random.randint(1,n_cli+1,2000),
    'fecha':       pd.date_range('2022-01-01','2024-06-30',periods=2000).strftime('%Y-%m-%d'),
    'total':       np.round(np.random.exponential(400,2000),2),
    'estado':      np.random.choice(['entregado','cancelado'],2000,p=[0.85,0.15]),
    'categoria':   np.random.choice(['Electrónica','Ropa','Hogar','Deportes'],2000),
})
clientes_df.to_sql('clientes', engine, if_exists='replace', index=False)
pedidos_df.to_sql('pedidos', engine, if_exists='replace', index=False)

# ── 2. Feature Engineering en SQL (una sola query) ──
query_features = """
    WITH stats AS (
        SELECT
            p.cliente_id,
            COUNT(*)                                                 AS n_pedidos,
            SUM(CASE WHEN p.estado='entregado' THEN p.total ELSE 0 END) AS gasto_total,
            AVG(p.total)                                             AS ticket_promedio,
            MAX(p.fecha)                                             AS ultima_compra,
            MIN(p.fecha)                                             AS primera_compra,
            SUM(CASE WHEN p.estado='cancelado' THEN 1 ELSE 0 END) * 1.0
                / COUNT(*)                                           AS tasa_cancelacion,
            COUNT(DISTINCT p.categoria)                              AS categorias_distintas,
            SUM(CASE WHEN p.fecha >= '2024-01-01' THEN 1 ELSE 0 END) AS pedidos_2024
        FROM pedidos p
        GROUP BY p.cliente_id
    ),
    con_rank AS (
        SELECT *,
               NTILE(4) OVER (ORDER BY gasto_total DESC) AS cuartil_gasto
        FROM stats
    )
    SELECT
        c.cliente_id, c.segmento, c.ciudad,
        COALESCE(r.n_pedidos, 0)             AS n_pedidos,
        COALESCE(r.gasto_total, 0)           AS gasto_total,
        COALESCE(r.ticket_promedio, 0)       AS ticket_promedio,
        COALESCE(r.tasa_cancelacion, 0)      AS tasa_cancelacion,
        COALESCE(r.categorias_distintas, 0)  AS categorias_distintas,
        COALESCE(r.pedidos_2024, 0)          AS pedidos_2024,
        COALESCE(r.cuartil_gasto, 4)         AS cuartil_gasto,
        CAST(julianday('2024-06-30') - julianday(c.fecha_registro) AS INTEGER)
                                             AS dias_desde_registro,
        -- Target: churn = no compró en los últimos 6 meses
        CASE WHEN r.ultima_compra < '2024-01-01' OR r.ultima_compra IS NULL
             THEN 1 ELSE 0 END               AS churn
    FROM clientes c
    LEFT JOIN con_rank r ON c.cliente_id = r.cliente_id
"""

df_ml = pd.read_sql(query_features, engine)
print(f'Dataset ML: {len(df_ml)} clientes, {len(df_ml.columns)} features')
print(f'Tasa de churn: {df_ml["churn"].mean()*100:.1f}%')
print(f'\nFeatures generadas por SQL:')
print(df_ml.dtypes.to_string())

# ── 3. Pipeline ML ──
num_cols = ['n_pedidos','gasto_total','ticket_promedio','tasa_cancelacion',
            'categorias_distintas','pedidos_2024','cuartil_gasto','dias_desde_registro']
cat_cols = ['segmento','ciudad']

X = df_ml[num_cols + cat_cols]
y = df_ml['churn']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

prep = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_cols),
])
pipe = Pipeline([('prep',prep),
    ('modelo',RandomForestClassifier(n_estimators=200,class_weight='balanced',random_state=42))])
pipe.fit(X_tr, y_tr)

pred  = pipe.predict(X_te)
proba = pipe.predict_proba(X_te)[:,1]
print('\n=== Resultados del modelo ===')
print(classification_report(y_te, pred, target_names=['Activo','Churn']))

# ── 4. Guardar predicciones en la BD ──
df_pred = df_ml.copy()
df_pred['prob_churn']   = pipe.predict_proba(X)[:,1]
df_pred['pred_churn']   = pipe.predict(X)
df_pred['riesgo']       = pd.cut(df_pred['prob_churn'],
    bins=[0,0.3,0.6,1.0], labels=['Bajo','Medio','Alto'])

# Guardar en la BD
df_pred[['cliente_id','prob_churn','pred_churn','riesgo']].to_sql(
    'predicciones_churn', engine, if_exists='replace', index=False
)
print('\n=== Predicciones guardadas en BD ===')
resumen = pd.read_sql(
    'SELECT riesgo, COUNT(*) as n, ROUND(AVG(prob_churn)*100,1) as prob_avg '
    'FROM predicciones_churn GROUP BY riesgo ORDER BY prob_avg DESC', engine
)
print(resumen.to_string(index=False))

### 🚀 Proyecto integrador — Análisis completo de negocio

Integrá todo lo aprendido en la Semana 21-22 en un proyecto end-to-end:

**Escenario:** Sos analista de datos en una empresa de retail. Tu tarea es construir un sistema de análisis y predicción.

**Paso 1 — BD (SQL):**
- Creá tablas: `clientes`, `productos`, `pedidos`, `detalle_pedido`
- Al menos 500 clientes, 100 productos, 2000 pedidos

**Paso 2 — Feature Engineering (SQL):**
- Usando una sola query con CTEs y window functions, generá al menos 10 features por cliente
- Incluí: recencia, frecuencia, valor monetario (RFM), tendencia de compra, diversificación

**Paso 3 — Modelos (scikit-learn):**
- Regresión: predecir el gasto del próximo trimestre
- Clasificación: predecir si comprará en los próximos 30 días

**Paso 4 — Resultados:**
- Guardar predicciones en la BD con `to_sql()`
- Reporte ejecutivo con métricas de modelo y segmentación de clientes
- Al menos una visualización con matplotlib o seaborn

In [ ]:
import pandas as pd, numpy as np
from sqlalchemy import create_engine
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, r2_score
import warnings; warnings.filterwarnings('ignore')

# 🚀 Tu proyecto integrador aquí:

# Paso 1 — BD:


# Paso 2 — Feature Engineering SQL:


# Paso 3 — Modelos:


# Paso 4 — Resultados:


## 📝 Reporte ejecutivo

- **Dataset:** 
- **Modelos entrenados:** 
- **Mejor métrica de regresión:** 
- **Mejor métrica de clasificación:** 
- **Segmentación de clientes:** 
- **Recomendaciones de negocio:** 

*(Doble click para editar)*

---
## ✅ Resumen de la Semana 21-22

### Lo que aprendiste

| Tema | Conceptos clave |
|------|-----------------|
| Regresión lineal | Pipeline, Ridge/Lasso, regularización, feature selection |
| Random Forest | Clasificación, class_weight, umbral, feature importance |
| SQL — Window Functions | ROW_NUMBER, RANK, LAG/LEAD, SUM/AVG OVER, NTILE |
| SQL — CTEs recursivos | Jerarquías, recurrencia, casos de uso |
| Integración SQL+ML | pd.read_sql, feature engineering en SQL, to_sql |

### ➡️ Próximos pasos — Semana 23-24
- Frameworks de Python para ciencia de datos: TensorFlow y Scikit-learn avanzado
- Proyectos finales de ciencia de datos con datasets reales

---

### 📚 Recursos recomendados
- [scikit-learn — Pipeline docs](https://scikit-learn.org/stable/modules/compose.html)
- [SQLite Window Functions](https://www.sqlite.org/windowfunctions.html)
- [Towards Data Science — Feature Engineering](https://towardsdatascience.com/feature-engineering)
- [Kaggle — Intermediate ML](https://www.kaggle.com/learn/intermediate-machine-learning)